<a href="https://colab.research.google.com/github/weiyu1053/IR_array_CNN/blob/main/%E3%80%8Cbeginner_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### 版權所有 2019 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# TensorFlow 2 新手快速入門

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/tutorials/quickstart/beginner"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/docs/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

這份簡短的入門教學會使用 [Keras](https://www.tensorflow.org/guide/keras/overview) 來：

1. 載入一個內建的資料集。
1. 建立一個能分類圖片的神經網路機器學習模型。
2. 訓練這個神經網路。
3. 評估模型的準確率。

這份教學是一本 [Google Colaboratory](https://colab.research.google.com/notebooks/welcome.ipynb) 筆記本。Python 程式會直接在瀏覽器中執行——這是學習和使用 TensorFlow 的好方法。要跟著這份教學操作，請點擊本頁最上方的按鈕，在 Google Colab 中開啟並執行這本筆記本。

1. 在 Colab 中連線到 Python 執行階段：在選單列的右上角，選擇 *連線（CONNECT）*。
2. 若要執行筆記本中的所有程式碼，請選擇 **執行階段（Runtime）** > **全部執行（Run all）**。若要一次執行一個程式碼儲存格，將滑鼠移到該儲存格上，點擊 **執行儲存格（Run cell）** 圖示。

![執行儲存格圖示](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/quickstart/images/beginner/run_cell_icon.png?raw=1)

## 設定 TensorFlow

首先，在程式中匯入 TensorFlow：

In [2]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


如果你不是在 [Colab](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb)，而是在自己的開發環境中操作，請參考[安裝指南](https://www.tensorflow.org/install)來設定 TensorFlow 開發環境。

注意：如果你使用自己的開發環境，請先確認已將 `pip` 升級到最新版本，才能安裝 TensorFlow 2 套件。詳情請見[安裝指南](https://www.tensorflow.org/install)。

## 載入資料集

載入並準備 MNIST 資料集（手寫數字 0～9 的圖片）。圖片的像素值範圍是 0 到 255。將這些數值除以 `255.0`，把範圍縮放到 0 到 1 之間。這同時也會把樣本資料從整數轉換成浮點數：

In [3]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## 建立機器學習模型

建立一個 `tf.keras.Sequential` 模型：

In [4]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)), #把 28×28 的二維陣列攤平成 784 維向量
  tf.keras.layers.Dense(128, activation='relu'), #全連接隱藏層，128 個神經元，激活函式用 ReLU
  tf.keras.layers.Dropout(0.2), #訓練時隨機關掉 20% 的神經元
  tf.keras.layers.Dense(10) #輸出層，10 個神經元
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[`Sequential`](https://www.tensorflow.org/guide/keras/sequential_model) 適合用來把多個層（layer）依序堆疊起來，其中每一層都只有一個輸入[張量（tensor）](https://www.tensorflow.org/guide/tensor)和一個輸出張量。層是具有已知數學結構、可以重複使用，並且帶有可訓練變數的函式。大多數 TensorFlow 模型都是由層組成的。這個模型使用了 [`Flatten`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten)、[`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense) 和 [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) 這三種層。

對於每一筆範例，模型會回傳一個由 [logits](https://developers.google.com/machine-learning/glossary#logits) 或 [log-odds（對數勝算）](https://developers.google.com/machine-learning/glossary#log-odds) 分數組成的向量，每個類別各對應一個分數。

In [5]:
predictions = model(x_train[:1]).numpy()
predictions

array([[ 7.6080638e-01, -1.6936553e-01, -6.2127972e-01, -1.8470296e-01,
        -2.0557800e-01,  4.7587544e-02, -2.9877961e-01, -7.4927592e-01,
        -6.9897994e-04,  7.1074271e-01]], dtype=float32)

`tf.nn.softmax` 函式會把這些 logits 轉換成每個類別的*機率*：

In [6]:
tf.nn.softmax(predictions).numpy()

array([[0.20449206, 0.08066923, 0.05133861, 0.07944142, 0.07780026,
        0.10021424, 0.07087679, 0.04517062, 0.09549021, 0.19450648]],
      dtype=float32)

注意：其實可以把 `tf.nn.softmax` 函式直接放進網路最後一層的激活函數（activation function）裡。雖然這樣能讓模型的輸出更直觀易懂，但並不建議這麼做，因為當模型輸出是 softmax 時，無法對所有模型都提供精確且數值穩定的損失計算。

使用 `losses.SparseCategoricalCrossentropy` 定義訓練用的損失函數：

In [7]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

損失函數會接收一個真實答案（ground truth）的向量和一個 logits 向量，並為每一筆範例回傳一個純量（scalar）損失值。這個損失值等於正確類別機率的負對數：如果模型完全確定正確的類別，損失就是零。

這個尚未訓練的模型給出的機率接近隨機猜測（每個類別約 1/10），所以初始損失應該接近 `-tf.math.log(1/10) ~= 2.3`。

In [8]:
loss_fn(y_train[:1], predictions).numpy()

np.float32(2.300445)

開始訓練之前，要先用 Keras 的 `Model.compile` 來設定並編譯模型。將 [`optimizer`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers)（最佳化器）類別設為 `adam`，將 `loss` 設為你前面定義的 `loss_fn` 函式，並把 `metrics` 參數設為 `accuracy`，指定要用準確率來評估模型。

In [9]:
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

## 訓練並評估模型

使用 `Model.fit` 方法來調整模型參數，讓損失降到最低：

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,312 (1.16 MB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 203,542 (795.09 KB)

In [10]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9128 - loss: 0.3022
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9568 - loss: 0.1447
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9685 - loss: 0.1047
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9724 - loss: 0.0882
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9764 - loss: 0.0744


`Model.evaluate` 方法用來檢查模型的表現，通常是在[驗證集（validation set）](https://developers.google.com/machine-learning/glossary#validation-set)或[測試集（test set）](https://developers.google.com/machine-learning/glossary#test-set)上進行。

In [11]:
model.evaluate(x_test,  y_test, verbose=2)

313/313 - 1s - 2ms/step - accuracy: 0.9779 - loss: 0.0761


[0.07605095207691193, 0.9779000282287598]

這個圖片分類器現在已在此資料集上訓練到約 98% 的準確率。想了解更多，請閱讀 [TensorFlow 教學](https://www.tensorflow.org/tutorials/)。

如果你希望模型直接回傳機率，可以把訓練好的模型包起來，並在後面接上 softmax：

In [12]:
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

In [13]:
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 10), dtype=float32, numpy=
array([[4.75470330e-10, 8.30500044e-11, 1.75711207e-07, 1.40683360e-05,
        1.45761944e-11, 3.83857355e-07, 5.82850622e-14, 9.99984860e-01,
        3.14636894e-09, 6.33193736e-07],
       [1.44747162e-06, 2.08062163e-04, 9.99406815e-01, 3.70436494e-04,
        2.00273097e-14, 2.83114582e-06, 4.03517674e-07, 7.12407910e-10,
        1.00730031e-05, 4.27985647e-09],
       [3.43561055e-07, 9.97979820e-01, 4.45324986e-05, 1.13967319e-04,
        1.40381999e-05, 1.89929779e-05, 3.05809444e-05, 9.78328520e-04,
        8.18941742e-04, 5.02905607e-07],
       [9.99644041e-01, 6.64383340e-11, 3.42972489e-05, 1.02717514e-07,
        6.33202490e-07, 4.88247097e-05, 2.39642650e-05, 9.92456989e-05,
        2.47572824e-08, 1.48927618e-04],
       [4.67248782e-08, 2.14029090e-11, 1.98531225e-07, 3.67391007e-09,
        9.97861445e-01, 1.56846070e-07, 1.30237837e-07, 1.96097426e-05,
        3.78689435e-07, 2.11812183e-03]], dtype=float32)>

## 結語

恭喜！你已經使用 [Keras](https://www.tensorflow.org/guide/keras/overview) API 和內建資料集訓練出一個機器學習模型了。

想看更多使用 Keras 的範例，請參考這些[教學](https://www.tensorflow.org/tutorials/keras/)。想深入了解如何用 Keras 建立模型，請閱讀這些[指南](https://www.tensorflow.org/guide/keras)。如果想學習更多關於載入與準備資料的內容，可以看看[載入圖片資料](https://www.tensorflow.org/tutorials/load_data/images)或[載入 CSV 資料](https://www.tensorflow.org/tutorials/load_data/csv)的教學。


## 第 3 步：實驗紀錄

每次只改一個地方，從建立模型那格（`Sequential`）重跑到 `evaluate`。

| 實驗 | 改了什麼 | 參數量 | 測試準確率 |
|---|---|---|---|
| 原始 | Dense(128), epochs=5 | 101,770 | 0.9779 |
| A | epochs=1 | | |
| B | epochs=10 | | |
| C | Dense(16) | | |
| D | Dense(512) | | |
| E | 拿掉 /255.0 正規化 | | |